# 02 — Local training validation

Runs the full ensemble training pipeline against synthetic data and validates the resulting ONNX artefact. Useful as a pre-flight check before submitting to AML.

In [ ]:
import sys, pathlib, json, tempfile
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))
from ml.train_ensemble import run
out_dir = tempfile.mkdtemp(prefix='aml_local_')
result = run(input_path=None, output_dir=out_dir, n_smoke=5_000)
print(json.dumps({k: v for k, v in result['metrics'].items() if isinstance(v, (int, float))}, indent=2))

In [ ]:
import onnxruntime as ort, numpy as np
from ml.train_ensemble import NUM_FEATURES, CAT_FEATURES
sess = ort.InferenceSession(f'{out_dir}/ensemble.onnx', providers=['CPUExecutionProvider'])
feed = {c: np.array([[0.0]], dtype=np.float32) for c in NUM_FEATURES}
feed.update({c: np.array([['SE']], dtype=object) for c in CAT_FEATURES})
sess.run(None, feed)

In [ ]:
result['fairness']